# AfroCare Stage 1 v2: DeBERTa pilot

Single-label classifier over 60 conditions plus OTHER. Pilot only: not for patient use until tested on clinician-reviewed real cases.

Generated cases help training but never count toward a score. Tuning reads validation data only; set `FINAL_EVAL = True` once, after tuning, to score the test split.

In [ ]:
TRIAL_ID = 'manual'
MODEL_NAME = 'microsoft/deberta-v3-small'
LEARNING_RATE = 2e-5
EPOCHS = 2
BATCH_SIZE = 32
MAX_LENGTH = 128
CLASS_WEIGHT_POWER = 0.5  # 0 is no weighting, 1 is fully balanced
SEED = 42
SMOKE_ROWS = 0  # above 0, trains on this many rows to check the notebook works
FINAL_EVAL = False
SAVE_MODEL = False

In [ ]:
TRIAL_ID = 'manual'

In [ ]:
from sklearn.metrics import f1_score


def score(df, preds, train_counts, thin_limit=100):
    scored = df.assign(pred=list(preds))
    scored = scored[scored['source'] != 'synthetic']
    labels = sorted(scored['label'].unique())
    f1s = f1_score(scored['label'], scored['pred'], labels=labels, average=None, zero_division=0)
    per_label = {label: float(f1) for label, f1 in zip(labels, f1s)}
    thin = [f1 for label, f1 in per_label.items() if train_counts.get(label, 0) < thin_limit]
    correct = scored['label'] == scored['pred']
    return {
        'rows': len(scored),
        'accuracy': float(correct.mean()),
        'macro_f1': float(sum(per_label.values()) / len(per_label)),
        'thin_macro_f1': float(sum(thin) / len(thin)) if thin else None,
        'by_source': {k: float(v) for k, v in correct.groupby(scored['source']).mean().items()},
        'per_label': per_label,
    }

In [ ]:
import glob
import json
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

torch.manual_seed(SEED)
np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

data_file = glob.glob('/kaggle/input/**/afrocare_dx_v2.csv', recursive=True)[0]
df = pd.read_csv(data_file, dtype=str)
labels = sorted(df['label'].unique())
label2id = {label: i for i, label in enumerate(labels)}

train = df[df['split'] == 'train']
val = df[df['split'] == 'val']
if SMOKE_ROWS:
    train = train.sample(min(SMOKE_ROWS, len(train)), random_state=SEED)
    val = val.sample(min(SMOKE_ROWS // 4, len(val)), random_state=SEED)
train_counts = train['label'].value_counts().to_dict()
print(device, len(df), len(train), len(val), len(labels))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=label2id).float().to(device)  # the checkpoint loads as 16-bit, which GradScaler rejects


def encode(texts):
    return tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors='pt')


def collate(batch):
    texts, targets = zip(*batch)
    return encode(texts), torch.tensor(targets)


counts = np.array([max(train_counts.get(label, 0), 1) for label in labels], dtype=float)
weights = (counts.sum() / (len(labels) * counts)) ** CLASS_WEIGHT_POWER
weights = torch.tensor(weights / weights.mean(), dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

loader = DataLoader(list(zip(train['text'], train['label'].map(label2id))), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
steps = len(loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(steps * 0.1), steps)
scaler = torch.amp.GradScaler(device, enabled=device == 'cuda')

started = time.time()
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for step, (inputs, targets) in enumerate(loader, 1):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device, enabled=device == 'cuda'):
            loss = loss_fn(model(**inputs).logits, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item()
        if step % 200 == 0:
            print(f'epoch {epoch + 1} step {step}/{len(loader)} loss {total / step:.4f} {time.time() - started:.0f}s', flush=True)
    print(f'epoch {epoch + 1} loss {total / len(loader):.4f}', flush=True)

In [ ]:
@torch.no_grad()
def predict(frame):
    model.eval()
    texts = frame['text'].tolist()
    found = []
    for start in range(0, len(texts), BATCH_SIZE * 2):
        inputs = encode(texts[start:start + BATCH_SIZE * 2]).to(device)
        with torch.amp.autocast(device, enabled=device == 'cuda'):
            found += model(**inputs).logits.argmax(-1).tolist()
    return [labels[i] for i in found]


result = score(val, predict(val), train_counts)
metrics = {
    "trial_id": TRIAL_ID,
    'val_macro_f1': result['macro_f1'],
    'val_accuracy': result['accuracy'],
    'val_thin_macro_f1': result['thin_macro_f1'],
    'val_by_source': result['by_source'],
    'val_rows': result['rows'],
    'train_seconds': round(time.time() - started),
}
pd.Series(result['per_label'], name='f1').sort_values().to_csv('val_per_label.csv')

if FINAL_EVAL:
    test = df[df['split'] == "test"]
    final = score(test, predict(test), train_counts)
    metrics['test'] = final

if SAVE_MODEL:
    model.save_pretrained('model')
    tokenizer.save_pretrained('model')

json.dump(metrics, open("metrics.json", 'w'), indent=2)
print(json.dumps(metrics, indent=2))